# 11.1 文档处理 (Document Processing)

RAG系统的第一步是将文档处理成可检索的单元。

本节涵盖：文本分块、向量化、索引构建

## 1. 文本分块

**目的**：将长文档切分为合适大小的片段

**基本原理**：分块大小影响检索精度——太大则信息不聚焦，太小则缺乏上下文。

**主要方法**：
- **固定大小分块**：按字符/token数切分，简单但可能切断语义
- **语义分块**：基于语义边界（段落、句子）切分
- **递归分块**：先按大单位切，超长时递归按小单位切

In [ ]:
import torch

torch.manual_seed(42)

text = 'Machine learning is a subset of AI. It learns from data. Deep learning uses neural networks. NLP processes language. Computer vision processes images. Each field has unique challenges and approaches.'

def fixed_chunk(text, chunk_size=30, overlap=5):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

def semantic_chunk(text):
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    return sentences

fixed_chunks = fixed_chunk(text, chunk_size=40, overlap=10)
semantic_chunks = semantic_chunk(text)

print('=== Text Chunking ===')
print(f'Original text ({len(text)} chars): {text[:60]}...')
print(f'\nFixed-size chunks (size=40, overlap=10):')
for i, c in enumerate(fixed_chunks):
    print(f'  Chunk {i}: "{c}"')
print(f'\nSemantic chunks (sentence-level):')
for i, c in enumerate(semantic_chunks):
    print(f'  Chunk {i}: "{c}"')
print(f'\nKey: Semantic chunks preserve meaning better.')
print(f'Overlap prevents losing information at chunk boundaries.')

## 2. 向量化与索引构建

**向量化**：将文本转换为密集向量表示，用于语义检索。
- 稠密向量：语义丰富，适合语义搜索（DPR, E5, BGE）
- 稀疏向量：精确匹配，适合关键词搜索（SPLADE）

**索引构建**：将向量组织为高效检索的数据结构。
- **FAISS**：Facebook的高效向量检索库，支持GPU
- **HNSW**：基于图的近似最近邻，查询速度快
- **倒排索引**：传统信息检索的索引方式

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DenseEncoder(nn.Module):
    def __init__(self, d_model=64, d_embed=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_embed))
    def forward(self, x):
        return F.normalize(self.encoder(x), dim=-1)

encoder = DenseEncoder()
n_docs = 200
docs = torch.randn(n_docs, 64)
query = torch.randn(1, 64)

doc_embeds = encoder(docs)
query_embed = encoder(query)

scores = query_embed @ doc_embeds.T
top_k = 5
top_indices = scores.topk(top_k, dim=-1).indices[0].tolist()
top_scores = scores.topk(top_k, dim=-1).values[0].tolist()

print('=== Dense Embedding & Indexing ===')
print(f'Documents: {n_docs}, Embedding dim: {doc_embeds.shape[-1]}')
print(f'Index size: {doc_embeds.numel() * 4 / 1e6:.2f} MB')
print(f'\nTop-{top_k} results:')
for i, (idx, score) in enumerate(zip(top_indices, top_scores)):
    print(f'  Rank {i+1}: doc={idx}, score={score:.4f}')

print(f'\n=== Index Comparison ===')
print(f'FAISS IVF: O(n/k) search, good for >1M vectors')
print(f'HNSW: O(log n) search, good for <10M vectors')
print(f'Brute force: O(n) search, exact results')
print(f'\nKey: Dense embeddings enable semantic search beyond keyword matching.')